In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
cd "/app/"

In [ ]:
%run env.py
%run run.py connect

In [ ]:
from vr4mice.analysis.analysis import style
import warnings

warnings.filterwarnings("ignore")

style()

In [ ]:
from vr4mice.schema.vr4mice import SignalsPhotodiode
from vr4mice.schema.latency_tests import SignalsPhotodiodeAligned, AllLatencies
from vr4mice.schema.inputs_videos import RawVideo, RawVideoSync, StepVideoFrame

# From DJ directly

In [ ]:
session = {"dataset": "Wiwaxia_2024-10-03_1"}

In [ ]:
RawVideo()

In [ ]:
RawVideo().populate(suppress_errors=True)

In [ ]:
RawVideo() & session

In [ ]:
RawVideoSync().populate(session)

In [ ]:
StepVideoFrame().populate(session)

In [ ]:
import pandas as pd

# Frame displayed at each game step, as an index in the raw OBS recording
step_time, frame_ids = (StepVideoFrame() & session).fetch1("step_time", "frame_ids")
frame_correspondance = pd.DataFrame({"step_time": step_time, "frame_ids": frame_ids})

In [ ]:
from vr4mice.schema.base_analysis import DataFrame
df = DataFrame().get_data(
    key={"dataset": "Wiwaxia_2024-10-03_1"},
    columns=[
        "dataset",
        "step_time",
        "reward",
        "mouse_in_left",
        "mouse_in_right",
        "trial",
        "aperture",
        "iti",
        "trial_left_choice",
        "trial_duration",
    ],
)

In [ ]:
merged_df = pd.merge_asof(df, frame_correspondance, left_on='step_time', right_on='step_time', direction="forward")

In [ ]:
merged_df[(merged_df["reward"] == 1)]

In [ ]:
# First rewarded step and the frame displayed then
reward_frame = int(merged_df.loc[merged_df["reward"] == 1, "frame_ids"].iloc[0])
reward_frame

In [ ]:
merged_df[merged_df["frame_ids"] == reward_frame]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(12, 6))
for axis, (label, frame) in zip(
    ax,
    [("Before", reward_frame - 4), ("Reward", reward_frame), ("After", reward_frame + 1)],
):
    axis.imshow(StepVideoFrame.get_frame(session, frame)[..., ::-1])  # BGR -> RGB
    axis.set_title(label)
    axis.axis("off")